## Modelado

In [ ]:
# ── 15. Modelo RandomForest ──────────────────────────────────────────────────
rf = SparkRFC(featuresCol="features", labelCol="label", seed=42)

# ── 16. ParamGrid ────────────────────────────────────────────────────────────
param_grid = (ParamGridBuilder()
    .addGrid(rf.numTrees, [10, 50, 100])
    .addGrid(rf.maxDepth, [5, 10, 15])
    .build())

# ── 17. Evaluador binario ────────────────────────────────────────────────────
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# ── 18. CrossValidator ───────────────────────────────────────────────────────
cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_auc,
    numFolds=3,
    parallelism=4,
    seed=42
)

# ── 19. Entrenamiento ────────────────────────────────────────────────────────
start_train = time.time()
cv_model = cv.fit(train_df)
elapsed_train_spark = time.time() - start_train

# ── 20. Predicción ───────────────────────────────────────────────────────────
start_pred = time.time()
predictions = cv_model.transform(test_df)
predictions.persist(StorageLevel.MEMORY_AND_DISK)
predictions.count()                          # materializa
elapsed_pred_spark = time.time() - start_pred

In [ ]:
# ── 23. Mejores hiperparámetros ───────────────────────────────────────────────
best_rf = cv_model.bestModel
best_params = {
    "numTrees": best_rf.getNumTrees,
    "maxDepth": best_rf.getOrDefault(best_rf.maxDepth)
}

# ── Matriz de confusión → métricas correctas (binario, sin MulticlassEvaluator)
cm_rows = (predictions
           .groupBy("label", "prediction")
           .count()
           .orderBy("label", "prediction"))
cm_rows.show()

# Extraer TP, TN, FP, FN como variables — solo 4 filas al driver
cm_dict = {(int(r["label"]), int(r["prediction"])): r["count"]
           for r in cm_rows.collect()}

TP = cm_dict.get((1, 1), 0)
TN = cm_dict.get((0, 0), 0)
FP = cm_dict.get((0, 1), 0)
FN = cm_dict.get((1, 0), 0)

accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)  if (TP + FP) > 0 else 0.0
recall    = TP / (TP + FN)  if (TP + FN) > 0 else 0.0
f1        = (2 * precision * recall / (precision + recall)
             if (precision + recall) > 0 else 0.0)

auc = evaluator_auc.evaluate(predictions)

print("══════════════════════════════════════════════════")
print("  CrossValidator (PySpark) — RandomForestClassifier")
print("══════════════════════════════════════════════════")
print(f"  Mejores parámetros : {best_params}")
print(f"  ROC AUC            : {auc:.4f}")
print(f"  Accuracy           : {accuracy:.4f}")
print(f"  Precision          : {precision:.4f}")
print(f"  Recall             : {recall:.4f}")
print(f"  F1-score           : {f1:.4f}")
print(f"  Tiempo entrenamiento (CV) : {elapsed_train_spark:.2f} s")
print(f"  Tiempo predicción         : {elapsed_pred_spark:.4f} s")
print("══════════════════════════════════════════════════")